# 🚗 Evaluate the Claude part of the voice assistant

This notebook runs the 118 test commands through **Claude alone** and through the **hybrid assistant** (offline model first, Claude for the rest), then rebuilds the charts and the demo with Claude connected.

**Time:** about 10 minutes. **Cost:** roughly 10–20 US cents of API usage with Claude Haiku 5.5.

Run the cells from top to bottom with `Shift + Enter`.

## 1. Add your API key (once)

1. Create a key by following [Get your Claude API key](https://platform.claude.com/docs/en/get-api-key).
2. In Colab, click the **🔑 key icon** in the left sidebar (*Secrets*) → **Add new secret**.
3. Name: `ANTHROPIC_API_KEY`, value: your key, and switch on **Notebook access**.

The key stays in your Colab account: it is never printed or saved in the project.

**Paste the key once**, with no spaces or line breaks.

## 2. Get the project and install it

In [ ]:
!git clone -q https://github.com/souadessemyhij1999/in-car-voice-assistant.git
%cd in-car-voice-assistant
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
raw = userdata.get("ANTHROPIC_API_KEY") or ""
parts = raw.split()                       # removes spaces/line breaks; catches a key pasted twice
key = parts[0] if parts else ""
os.environ["ANTHROPIC_API_KEY"] = key
if len(parts) > 1:
    print("Note: the secret contained more than one value (key pasted twice?) - using only the first one.")
print("API key loaded:", key.startswith("sk-ant-"), "| length:", len(key))   # the key itself is never printed

## 3. Quick check: tests and one live command

In [ ]:
!python -m pytest -q tests
!python src/assistant.py --text "I'm freezing, and take me home please"

## 4. Run the evaluation (≈ 5–8 minutes)

In [ ]:
!python src/evaluate.py --backend local  > /dev/null
!python src/evaluate.py --backend claude | tail -25
!python src/evaluate.py --backend hybrid | tail -25

## 5. Results

In [ ]:
!python src/report.py
from IPython.display import Image, Markdown, display
display(Markdown(open("results/comparison.md").read()))
display(Image("assets/success_by_category.png"))

## 6. Rebuild the demo with Claude connected

In [ ]:
!python src/demo.py
display(Image("assets/demo_last_frame.png"))

## 7. Download the results

This creates `llm_results.zip`. **Upload it in your chat with Claude**, and the README and GitHub repository will be updated with the measured LLM numbers.

In [ ]:
!zip -qr llm_results.zip results assets
from google.colab import files
files.download("llm_results.zip")